### Initialize

In [61]:
import os
import spotipy
from spotipy.cache_handler import CacheFileHandler
from spotipy.oauth2 import SpotifyOAuth

playset = dict()
known_tracks = dict()
known_artists = dict()
all_artists = set()

### (Re-)Authenticate user

In [64]:
cache_handler = CacheFileHandler(cache_path=".spotipy")
os.remove(cache_handler.cache_path)

auth_manager = SpotifyOAuth(
    scope="user-library-read playlist-read-private",
    client_id="",
    client_secret="",
    redirect_uri="http://localhost:8080",
    show_dialog=True,
    cache_handler=cache_handler,
)
sp = spotipy.Spotify(auth_manager=auth_manager)

current_user = sp.current_user()
print("Logged in as: " + current_user["display_name"])

Logged in as: Bobin Belloner


### Get user's tracks

In [65]:
known_tracks[current_user['id']] = set()
known_artists[current_user['id']] = set()

# get liked tracks
results = sp.current_user_saved_tracks(limit=50)
while results:
    for track in results['items']:
        track = track['track']
        playset[track['id']] = track
        known_tracks[current_user['id']].add(track['id'])
        known_artists[current_user['id']].add(track['artists'][0]['id'])
        all_artists.add(track['artists'][0]['id'])
    if results['next']:
        results = sp.next(results)
    else:
        results = None

# get tracks from users playlists
playlists = set()
results = sp.current_user_playlists(limit=50)
while results:
    for playlist in results['items']:
        playlists.add(playlist['id'])
    if results['next']:
        results = sp.next(results)
    else:
        results = None

for playlist in playlists:
    results = sp.playlist_tracks(playlist)
    while results:
        for track in results['items']:
            track = track['track']
            playset[track['id']] = track
            known_tracks[current_user['id']].add(track['id'])
            known_artists[current_user['id']].add(track['artists'][0]['id'])
            all_artists.add(track['artists'][0]['id'])
        if results['next']:
            results = sp.next(results)
        else:
            results = None

### Filter and enrich playset

In [66]:
# Find the tracks all players know
k = 2 # how many players need to know a track
tracks_known_by_k = set()

for id, track in playset.items():
    known_by = 0
    for player in known_tracks.keys():
        if track['id'] in known_tracks[player]:
            known_by += 1
    if known_by >= k:
        tracks_known_by_k.add(track['id'])

playset = dict((k, v) for k, v in playset.items() if k in tracks_known_by_k)

In [67]:
# add tracks by artists that are known by k players
k = 2 # how many players need to know an artist
artists_known_by_k = set()

for artist in all_artists:
    known_by = 0
    for player in known_artists.keys():
        if artist in known_artists[player]:
            known_by += 1
    if known_by >= k:
        artists_known_by_k.add(artist)#

for artist in artists_known_by_k:
    results = sp.artist_top_tracks(artist, country='DE')
    tracks = results['tracks']
    for track in tracks:
        playset[track['id']] = track

### Output playset

In [68]:
for track in playset.values():
    print(track['name'], ' - ', track['artists'][0]['name'])

atma  -  ok37
heat  -  ok37
Blood Walk  -  Cabaret Nocturne
HiTek  -  Insert Name Here
Green Karma - Original Mix  -  Cabaret Nocturne
Healing  -  Massano
Reflector  -  Booka Shade
Now Here No Where  -  Kölsch
Motherboard  -  Ben Böhmer
Invitation (feat. Kodie Shane)  -  Ashnikko
88  -  RYSY
Montezuma  -  RYSY
Massive Bells  -  Cores
Shaded  -  biskuwi
Dance With Me - Kölsch Remix  -  Kevin de Vries
NRG  -  RYSY
Fuck Me in Shibuya  -  Ashnikko
Cry (feat. Grimes)  -  Ashnikko
Miss Nectarine  -  Ashnikko
Super Soaker (feat. Daniela Lalita)  -  Ashnikko
Moonlight Magic  -  Ashnikko
WEEDKILLER  -  Ashnikko
Worms  -  Ashnikko
Cheerleader  -  Ashnikko
Slumber Party (feat. Princess Nokia)  -  Ashnikko
Deal With It (feat. Kelis)  -  Ashnikko
Daisy  -  Ashnikko
Halloweenie V: The Moss King  -  Ashnikko
Off Grid  -  Thomas Helmich
Toxic  -  Ashnikko
Tokyo Nights  -  Kevin de Vries
Perception  -  Chris Avantgarde
Pegasus  -  Kevin de Vries
Daisy 2.0 (feat. Hatsune Miku)  -  Ashnikko
Sass Pancakes